## Project WorkFlow For Real Estate Intelligence System


Note:This is not detailed code just some snippets from the whole project

### Part 1 Data set

There is 3 datasets 
Listings, including full descriptions and average review score

Reviews, including unique id for each reviewer and detailed comments

Calendar, including listing id and the price and availability for that day

We made some preprocessing according to each part of the project

### 2- Machine Learing Price Prediction

In [ ]:
#preProcessing
#Keeping only some of the columns because the dataset was more than 90 columns
columns_to_keep_cleansed = [
    'price', 'neighbourhood_group_cleansed',
    'latitude', 'longitude', 'property_type', 'room_type', 'accommodates',
    'bathrooms', 'bedrooms', 'beds', 'bed_type', 'amenities', 'square_feet',
    'security_deposit', 'cleaning_fee', 'guests_included', 'extra_people',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    'host_is_superhost', 'host_listings_count', 'host_total_listings_count'
]


df_pp.isnull().sum()

financial_cols = ['price', 'security_deposit', 'cleaning_fee', 'extra_people']

for col in financial_cols:
    df_pp[col] = df_pp[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)#force it to use replace not regex
    df_pp[col] = pd.to_numeric(df_pp[col], errors='coerce') # errors coerce will turn the value into nan if they can't convert to number

df_pp['security_deposit'] = df_pp['security_deposit'].fillna(0)
df_pp['cleaning_fee'] = df_pp['cleaning_fee'].fillna(0)

df_pp.isnull().sum()




In [ ]:
#columns encoding 
#detecting top animents and convert it one hot encoding 

top_amenities = [
    'Air Conditioning', 
    'Kitchen', 
    'Free Parking on Premises', 
    'Wireless Internet', 
    'Washer', 
    'Dryer', 
    'Hot Tub', 
    'Pool', 
    'Gym', 
    'Indoor Fireplace', 
    'Elevator in Building',
    'Pets Allowed',
    'Family/Kid Friendly',
    'Wheelchair Accessible',
    'TV'
]

for amenity in top_amenities:
    col_name = f"has_{amenity.replace(' ', '_').replace('/', '_')}" # Clean slashes for column names
    df_pp[col_name] = df_pp['amenities'].str.contains(amenity, na=False).astype(int)


df_pp['amenities_count'] = df_pp['amenities'].apply(lambda x: len(str(x).split(',')))


df_pp = df_pp.drop(columns=['amenities'])

categorical_cols = [
    'neighbourhood_group_cleansed', 
    'property_type', 
    'room_type', 
    'bed_type'
]

df_pp = pd.get_dummies(df_pp, columns=categorical_cols, drop_first=True)

In [ ]:
#modeling 
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

predictions = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mse = mean_squared_error(y_test, predictions)

#future work for improving 
#feature engineering 
importances = xgb_model.feature_importances_

### 3-Deel Learninig Sentiment Analysis

In [ ]:
#preprocessing for reviews dataset

#merging between the reveiws dataset and the listings to get the review scores for 
#sentiment labels join them on listings_id

listings_subset = df_listings[['id', 'review_scores_rating']].copy()
reviews_subset = df_reviews[['listing_id', 'comments']].copy()

df_merged = pd.merge(
    reviews_subset, 
    listings_subset, 
    left_on='listing_id', 
    right_on='id', 
    how='inner'
)

#showing the distribution and balance of the review scores
sns.set_theme(style="whitegrid")

# Create a figure with two subplots (Histogram on top, Boxplot on bottom)
fig, (ax_box, ax_hist) = plt.subplots(2, sharex=True, figsize=(10, 8), gridspec_kw={"height_ratios": (.15, .85)})

# 1. Top Subplot: Boxplot (Great for spotting outliers)
sns.boxplot(x=df_merged['review_scores_rating'], ax=ax_box, color='lightskyblue')
ax_box.set(xlabel='') # Hide the x-label for the top plot
ax_box.set_title('Distribution of Review Scores', fontsize=16)

# 2. Bottom Subplot: Histogram with KDE (Density curve)
sns.histplot(df_merged['review_scores_rating'], bins=30, kde=True, ax=ax_hist, color='royalblue', edgecolor='black')
ax_hist.set_xlabel('Review Score Rating (0 - 100)', fontsize=12)
ax_hist.set_ylabel('Number of Reviews', fontsize=12)

# Add a vertical line showing the median score
median_score = df_merged['review_scores_rating'].median()
ax_hist.axvline(median_score, color='red', linestyle='--', linewidth=2, label=f'Median: {median_score}')
ax_hist.legend()

plt.tight_layout()
plt.show()

#some text cleaninig 
def clean_text(text):
    text = re.sub(r"http\S+", "", text)  
    text = re.sub(r"@\w+", "", text)     
    text = re.sub(r"#\w+", "", text)     
    text = re.sub(r"[^\w\s.,!?]", "", text)  
    return text

df_merged["comments"] = df_merged["comments"].apply(clean_text)
df_merged.head()

#calculating the word counts for padding and for the vocab
df_merged["word_count"] = df_merged["comments"].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 6))
sns.histplot(df_merged["word_count"], bins=50, kde=True)

plt.title("Word Count Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

print("90%:", df_merged["word_count"].quantile(0.90))
print("95%:", df_merged["word_count"].quantile(0.95))
print("99%:", df_merged["word_count"].quantile(0.99))

#splitting the dataset
X = df_balanced['comments'].values
y = df_balanced['sentiment_label'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(
    df_balanced['comments'], 
    df_balanced['sentiment_label'], 
    test_size=0.15, 
    random_state=42, 
    stratify=df_balanced['sentiment_label'] # Ensures balance is kept in the split
)


X_train, X_val, y_train, y_test_val = train_test_split(
    X_train_val, 
    y_train_val, 
    test_size=0.15, 
    random_state=42,
    stratify=y_train_val
)

In [ ]:
# BILSTM
max_vocab_size = 20000  
max_sequence_length = 200
embedding_dim = 128

vectorizer = TextVectorization(
    max_tokens=max_vocab_size,
    output_mode='int',
    output_sequence_length=max_sequence_length
)

model = Sequential([
    
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    
    
    Embedding(input_dim=max_vocab_size, output_dim=embedding_dim, mask_zero=False),
    
   
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    
   
    Dense(32, activation='relu'),
    Dropout(0.5),
    
   
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
#DistilBert
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


training_args = TrainingArguments(
    output_dir="/kaggle/working/airbnb_sentiment_model",
    eval_strategy="epoch",           
    save_strategy="epoch",           
    logging_strategy="epoch",        
    learning_rate=2e-5,
    per_device_train_batch_size=32, 
    per_device_eval_batch_size=32,
    num_train_epochs=5,              
    weight_decay=0.01,
    fp16=True,                    
    load_best_model_at_end=True,    
    metric_for_best_model="loss",    
    greater_is_better=False,         
    report_to="none"              
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)] 
)

In [ ]:
#saving the model and to use it as custom embedding to our vector database
save_directory = "./my_real_estate_distilbert"

trainer.save_model(save_directory)

tokenizer.save_pretrained(save_directory

### Data Ingestion

In [ ]:
COLLECTION_NAME = "seattle_airbnb_inventory"

#initializing the custom embedder
class DistilBERTMeanPoolingEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_path: str):
        print(f"Loading local model and tokenizer from {model_path}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path)
        
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        print(f"Model loaded successfully on {self.device}.")

    def __call__(self, input: Documents) -> Embeddings:
        encoded_input = self.tokenizer(
            input, 
            padding=True, 
            truncation=True, 
            return_tensors='pt',
            max_length=512,
            return_token_type_ids=False 
        ).to(self.device)

       
        with torch.no_grad():
            
            encoded_input.pop("token_type_ids", None) 
            
            model_output = self.model(**encoded_input)

        
        token_embeddings = model_output[0] 
        attention_mask = encoded_input['attention_mask']

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        # 4. Return as a standard Python list for ChromaDB
        return mean_pooled.cpu().numpy().tolist()

In [ ]:
#identitfying the structure of the vector database
df = pd.read_csv(CSV_DATA_PATH)

documents = df['comments'].tolist()

ids = df['chroma_review_id'].astype(str).tolist()

metadata_cols = [col for col in df.columns if col not in ['comments', 'chroma_review_id']]

metadatas = df[metadata_cols].to_dict(orient='records')

In [ ]:
#starting the data ingestion
chroma_client = PersistentClient(path=CHROMA_DB_PATH)


embedding_func = DistilBERTMeanPoolingEmbeddingFunction(MODEL_DIR)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_func,
    metadata={"hnsw:space": "cosine"} 
)

### Rag Engine

In [ ]:
#initializing the embedder , vectordatabase and the LLM
self.embedder = DistilBERTMeanPoolingEmbeddingFunction(self.model_dir)

        
self.chroma_client = PersistentClient(path=self.chroma_path)
self.collection = self.chroma_client.get_collection(
name=self.collection_name,
embedding_function=self.embedder
)

        
self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3, max_retries=3)

In [ ]:
#prompt engineering and defining the prompt template for langchain
 # 4. Define Prompt Template
        template = """
        You are an expert, highly professional Real Estate and Airbnb Agent in Seattle. 
        A client has asked you a question. 
        
        You must answer their question based ONLY on the following property database context. 
        Do not invent properties, prices, or amenities that are not in the context.
        If the context does not contain relevant properties to answer the question, politely inform the client.
        
        When recommending a property, always mention the Listing ID, the price, and why it fits their request based on the review.

        PROPERTY DATABASE CONTEXT:
        {context}

        CLIENT QUESTION: 
        {question}
        
        AGENT RESPONSE:
        """
        self.prompt = ChatPromptTemplate.from_template(template)
        self.rag_chain = self.prompt | self.llm | StrOutputParser()

        

In [ ]:
#retreival formatting and generation
def get_recommendation(self, user_query: str, n_results: int = 5) -> dict:
        """Handles retrieval, formatting, and generation."""
        # Retrieve
        results = self.collection.query(query_texts=[user_query], n_results=n_results)
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        
        # Format Context
        context_string = ""
        for i in range(len(documents)):
            meta = metadatas[i]
            context_string += f"\n--- Property Option {i+1} ---\n"
            context_string += f"Listing ID: {meta.get('listing_id', 'Unknown')}\n"
            context_string += f"Price: ${meta.get('price', 'Unknown')}/night\n"
            context_string += f"Bedrooms: {meta.get('bedrooms', 'Unknown')}\n"
            context_string += f"Neighborhood: {meta.get('neighbourhood_cleansed', 'Unknown')}\n"
            context_string += f"Review Snippet: {documents[i]}\n"

        # Generate
        response = self.rag_chain.invoke({
            "context": context_string,
            "question": user_query
        })
        
        return {
            "agent_response": response,
            "raw_context": context_string
        }

### FastAPI main.py

In [ ]:
#initializing classes
class ChatRequest(BaseModel):
    query: str
    n_results: int = 5 

class ChatResponse(BaseModel):
    agent_response: str
    raw_context: str


app = FastAPI(
    title="Seattle Real Estate RAG API",
    description="An intelligent property recommendation engine powered by DistilBERT, ChromaDB, and Google Gemini.",
    version="1.0.0"
)

In [ ]:
#on startup 
@app.on_event("startup")
async def startup_event():
    """Fires when the server starts to load the heavy ML models into memory."""
    global rag_service
    try:
        rag_service = RealEstateRAG()
    except Exception as e:
        print(f"Failed to initialize RAG Engine: {e}")

In [ ]:
#post endpoint
async def get_recommendation(request: ChatRequest):
    """
    Accepts a natural language query and returns AI-generated property recommendations.
    """
    if not rag_service:
        raise HTTPException(status_code=503, detail="AI Engine is still initializing or offline.")
    
    try:
        
        result = rag_service.get_recommendation(
            user_query=request.query, 
            n_results=request.n_results
        )
        return ChatResponse(
            agent_response=result["agent_response"],
            raw_context=result["raw_context"]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

### streamLit

In [ ]:
# ==========================================
# 1. Page Configuration & CSS
# ==========================================
st.set_page_config(
    page_title="Seattle AI Real Estate Agent",
    page_icon="🏠",
    layout="centered"
)

# A little custom CSS to make the chat look cleaner
st.markdown("""
<style>
    .stChatMessage {border-radius: 10px;}
    .stChatInputContainer {padding-bottom: 20px;}
</style>
""", unsafe_allow_html=True)

In [ ]:
# Call the FastAPI Backend
    with st.chat_message("assistant"):
        # We use a spinner so the user knows the AI is "thinking"
        with st.spinner("Searching the database and analyzing reviews..."):
            
            try:
                # Send the POST request to your FastAPI server
                response = requests.post(
                    API_URL, 
                    json={"query": prompt, "n_results": 5},
                    timeout=120 
                )
                
                if response.status_code == 200:
                    data = response.json()
                    agent_reply = data["agent_response"]
                    raw_context = data["raw_context"]
                    
                    # Display the AI's response
                    st.markdown(agent_reply)
                    
                    # Optional: Add an expander to let users see the raw RAG data!
                    with st.expander("🔍 View Raw Database Context"):
                        st.text(raw_context)
                        
                    # Save the response to state
                    st.session_state.messages.append({"role": "assistant", "content": agent_reply})
                    
                else:
                    error_msg = f"⚠️ Backend Error: {response.status_code} - {response.text}"
                    st.error(error_msg)
                    
            except requests.exceptions.ConnectionError: